# Causal control: control_content_matched, seed 2025

Same audited HVPD-DVCS group-disjoint split, frozen DINOv3-L/16, 256px, AdamW, early stop, checkpoint and held-out test protocol. No local GPU.


## Kaggle GPU setup

In Settings choose Accelerator GPU T4 x2 and enable Internet when weights or the dataset have not already been cached. Two T4 GPUs have separate 16 GB VRAM, not a shared 32 GB memory pool. Train on device cuda:0. Save Version > Save & Run All stores the output; interrupted unsaved versions can lose progress.


In [ ]:
# Kaggle-only dependency preparation; no Windows C: installation.
from pathlib import Path
import os,sys,subprocess,importlib.util
KAGGLE_PROJECT=Path('/kaggle/working/salf_dino')
for relative in ['cache/tmp','cache/pip','cache/huggingface','cache/torch','cache/matplotlib','checkpoints','dataset','results']:
    (KAGGLE_PROJECT/relative).mkdir(parents=True,exist_ok=True)
for name,rel in {'TEMP':'cache/tmp','TMP':'cache/tmp','TMPDIR':'cache/tmp',
 'PIP_CACHE_DIR':'cache/pip','HF_HOME':'cache/huggingface','TORCH_HOME':'cache/torch',
 'MPLCONFIGDIR':'cache/matplotlib'}.items():
    os.environ[name]=str(KAGGLE_PROJECT/rel)
required=[(name,pkg) for name,pkg in [('imagehash','ImageHash>=4.3'),
 ('timm','timm>=1.0.25'),('huggingface_hub','huggingface_hub>=0.30')]
 if importlib.util.find_spec(name) is None]
try:
    from importlib.metadata import version
    from packaging.version import Version
    if importlib.util.find_spec('timm') and Version(version('timm'))<Version('1.0.25'):
        required.append(('timm','timm>=1.0.25'))
except Exception as e:print('Package version audit:',type(e).__name__)
if required:
    packages=list(dict.fromkeys(v for _,v in required))
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--disable-pip-version-check']+packages)
print('Kaggle runtime Python:',sys.executable)


In [ ]:
# Project-specific caches set before torch/timm/huggingface imports.
from pathlib import Path
import os
ROOT=Path('/kaggle/working/salf_dino')
for key,rel in {'TEMP':'cache/tmp','TMP':'cache/tmp','PIP_CACHE_DIR':'cache/pip',
 'HF_HOME':'cache/huggingface','HF_HUB_CACHE':'cache/huggingface/hub',
 'TORCH_HOME':'cache/torch','MPLCONFIGDIR':'cache/matplotlib',
 'IPYTHONDIR':'cache/ipython','PYTHONPYCACHEPREFIX':'cache/pycache'}.items():
    path=ROOT/rel;path.mkdir(parents=True,exist_ok=True)
    os.environ[key]=str(path)

import os, random, re, io, json, math, time, warnings, hashlib, zipfile, shutil, gc
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             precision_recall_fscore_support, confusion_matrix, classification_report)
import imagehash
import timm
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

# On Windows, use a folder on D: as requested; other environments use current workspace.
ROOT = Path('/kaggle/working/salf_dino')
DATA_DIR = ROOT / 'dataset'
EXTRACT_DIR = DATA_DIR / 'DVCS'
RESULT_DIR = ROOT / 'results'
for p in [DATA_DIR, EXTRACT_DIR, RESULT_DIR]: p.mkdir(parents=True, exist_ok=True)

DATASET_ID = 'WieszczUrsynowa/HVPD-HarmfulVisualPlotsDataset'
MODE = 'paper'  # 256px frozen-encoder benchmark; not original full finetuning
RES = 256  # 8GB frozen-encoder protocol
DINO_NAME = 'vit_large_patch16_dinov3.lvd1689m' if MODE == 'paper' else 'vit_base_patch16_dinov3.lvd1689m'
SWIN_NAME = 'swinv2_tiny_window16_256.ms_in1k'
EPOCHS = 20 if MODE == 'paper' else 2
SEEDS = [42, 123, 2025] if MODE == 'paper' else [42]
BATCH_SIZE = 2 if MODE == 'paper' else 4
GRAD_ACCUM = 8 if MODE == 'paper' else 1
RUN_TRAINING = True  # PILOT TRAINING ON GPU; audited labels and group split gate all training
SELECTED_MODEL = 'control_content_matched'  # model per saved Kaggle version
SELECTED_SEED = 2025
MAX_EPOCHS_THIS_VERSION = 20  # whole experiment within this Kaggle version
RUN_ABLATIONS = False
DOWNLOAD_DATA = True  # False if DVCS.zip already exists
USE_ROBUST_TRAIN_AUG = True  # SAME train augmentation for every model (no confounding)
NUM_WORKERS = 0  # avoid prior DataLoader cleanup multiprocessing warnings
PHASH_MAX_DISTANCE = 4  # experimenter-defined; NOT verified as exact paper threshold
SEED_SPLIT = 42
FORCE_RESPLIT = False
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if RUN_TRAINING and DEVICE.type!='cuda':
    raise RuntimeError('Enable Settings > Accelerator > GPU T4 x2.')
if DEVICE.type=='cpu':print('CPU audit mode. No training until CUDA is enabled.')
print('Root:', ROOT, '| MODE:', MODE, '| DEVICE:', DEVICE)
if DEVICE.type == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print('GPU:', torch.cuda.get_device_name(0), '| total VRAM GB:', round(total/2**30,1))
    if MODE == 'paper' and total < 20 * 2**30:
        print('8GB: FROZEN DINOv3-L protocol; NOT faithful full fine-tuning')
if RUN_TRAINING and MODE == 'paper' and DEVICE.type != 'cuda':
    raise RuntimeError('Publication-grade mode requires a CUDA-capable GPU in this notebook.')
assert MODE in ('pilot','paper') and RES % 16 == 0

In [ ]:
# Fresh proposed SALF-DINO seed42 experiment. No private old baseline checkpoint is restored.
from pathlib import Path
PREVIOUS_VERSION_ROOT = None
print('FRESH_CLOUD_RUN: proposed_salf seed 42; no conflicting baseline checkpoint')


## 2. Download and inspect the exact DVCS archive

Download only `DVCS.zip` (~1.19 GB on the currently published repo), not CIS.zip (~10 GB). The live file size and image count may change. Inspect ZIP members before extraction. No web scraping is used to create the main benchmark.

In [ ]:
zip_path = DATA_DIR / 'DVCS.zip'
if not zip_path.exists() and PREVIOUS_VERSION_ROOT is not None:
    prior=PREVIOUS_VERSION_ROOT/'dataset'/'DVCS.zip'
    if prior.exists():zip_path=prior
if not zip_path.exists():
    if not DOWNLOAD_DATA:
        raise FileNotFoundError(f'Copy DVCS.zip to {zip_path} or set DOWNLOAD_DATA=True')
    downloaded = hf_hub_download(repo_id=DATASET_ID, repo_type='dataset', filename='DVCS.zip', local_dir=str(DATA_DIR))
    zip_path = Path(downloaded)
assert zipfile.is_zipfile(zip_path), f'Not a valid ZIP: {zip_path}'
with zipfile.ZipFile(zip_path) as z:
    members = [i for i in z.infolist() if not i.is_dir()]
    print('ZIP members:', len(members), 'uncompressed GB:', round(sum(i.file_size for i in members)/2**30,2))
    print('Sample archive paths:')
    for info in members[:25]: print('  ', info.filename)
    ext_counts = Counter(Path(i.filename).suffix.lower() for i in members)
    print('Extensions:', ext_counts.most_common(12))

In [ ]:
# Zip-slip-safe unpack; skip already extracted members.
with zipfile.ZipFile(zip_path) as z:
    base = EXTRACT_DIR.resolve()
    for info in tqdm(z.infolist(), desc='Extract DVCS'):
        if info.is_dir() or info.filename.startswith('__MACOSX/'): continue
        dest = (EXTRACT_DIR / info.filename).resolve()
        if dest != base and base not in dest.parents:
            raise ValueError(f'Unsafe ZIP member: {info.filename}')
        if dest.exists() and dest.stat().st_size == info.file_size: continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as src, dest.open('wb') as dst: shutil.copyfileobj(src, dst)
IMAGE_EXTS={'.png','.jpg','.jpeg','.gif','.webp','.bmp','.tif','.tiff'}
image_paths = sorted(p for p in EXTRACT_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
print('Extracted image count:', len(image_paths))
assert len(image_paths)>0, 'No images found. Inspect archive nesting/format before proceeding.'

## 3. Label and category audit — never invent labels

Archive directory structures have not been independently audited, so labels are resolved only from unambiguous names **or** a manual `manual_labels.csv` manifest with `relative_path,label,category,series_id` columns. `label`: 0 = non-deceptive, 1 = deceptive. Put the manifest in `dataset/`. Unresolved paths are exported and execution stops; inspect the files and edit the manifest, then rerun this cell. `series_id` is optional but important to prevent leakage of different renderings from the same underlying numerical series.

In [ ]:
# Provenance-aware HVPD-DVCS labels, validated against the published 24,480-image inventory.
# Public DVCS.zip also contains 187 separate web-chart examples (hash-qualified names).
# Exclude these extras for an apples-to-apples 24,480-item synthetic dataset comparison.
CATEGORIES=['srtAxY','twoYaxes','3dpie','colNoScl','trdStck']
EXPECTED={
 'srtAxY':{0:4200,1:4200},
 'twoYaxes':{0:512,1:2046},
 '3dpie':{0:3072,1:1016},
 'colNoScl':{0:0,1:2560},
 'trdStck':{0:4646,1:2228}
}
def parse_dvcs_relative_path(rel):
    fname=Path(rel).name
    stem=Path(fname).stem
    m=re.match(r'^(chart_)?(srtAxY|twoYaxes|3dpie|colNoScl|trdStck)_(.+)$',stem,re.I)
    if not m:
        raise ValueError('Unrecognized DVCS file: '+str(rel))
    deceptive=0 if m.group(1) else 1
    canon=next(x for x in CATEGORIES if x.lower()==m.group(2).lower())
    suffix=m.group(3)
    # Web-scraped examples have 6-digit ID + 16-character hash; not part of published DVCS counts.
    if deceptive and re.match(r'^\d{6}_[0-9a-f]{16}_',suffix,re.I):
        return None
    if canon=='colNoScl':
        q=re.match(r'^(\d+)_(\d+)_(?:PY|VBA)$',suffix,re.I)
        if not q:raise ValueError('Unexpected colNoScl synthetic pattern: '+fname)
        series_id=f'colNoScl_{int(q.group(1))}_{int(q.group(2))}'
    else:
        q=re.match(r'^(\d+)(?:_|$)',suffix,re.I)
        if not q:raise ValueError('Unexpected synthetic family ID: '+fname)
        series_id=f'{canon}_{int(q.group(1))}'
    return dict(relative_path=Path(rel).as_posix(),label=deceptive,
                category=canon,series_id=series_id)

records=[];excluded=[]
for full_path in image_paths:
    rel=full_path.relative_to(EXTRACT_DIR)
    parsed=parse_dvcs_relative_path(rel)
    if parsed is None:excluded.append(rel.as_posix())
    else:records.append(parsed)
df=pd.DataFrame(records)
assert len(df)==24480,(len(df),len(excluded))
assert len(excluded)==187,(len(excluded),excluded[:8])
assert df.relative_path.is_unique
assert df['label'].value_counts().to_dict()=={0:12430,1:12050}
for cat,classes in EXPECTED.items():
    for lab,count in classes.items():
        actual=len(df[(df.category==cat)&(df.label==lab)])
        assert actual==count,(cat,lab,actual,count)
assert df.series_id.notna().all() and not df.series_id.eq('').any()
# Same underlying generated series never goes to different split, even across label/style variants.
df.to_csv(DATA_DIR/'label_audit.csv',index=False)
(DATA_DIR/'excluded_web_examples.txt').write_text('\n'.join(excluded),encoding='utf-8')
print('PUBLIC HVPD DVCS VERIFIED:',len(df),'synthetic images, 187 excluded web examples')
print('PER CLASS:',df['label'].value_counts().to_dict())
print('CATEGORY x LABEL:\n',pd.crosstab(df.category,df.label))
print('UNIQUE SOURCE SERIES:',df.series_id.nunique())
print('Manual labels were NOT inferred from class-specific adjectives.')


## 4. Group-disjoint split with pHash duplicate checks

Near-duplicate detection uses pHash with a disclosed threshold and connected components. This is an independent split (not the author's unreleased exact split). When a true numerical-series identifier is present in `series_id`, it is merged into the grouping. **Without series IDs, distinct renderings from the same source may still leak.** We record exact split CSV and SHA-256 manifest hash.

In [ ]:
# Leakage control: group by verified numerical-series identity and exact SHA-256 duplicates.
# Do NOT transitively merge near-equal perceptual hashes: it collapses unrelated synthetic charts
# into giant connected components and can produce train=24474, val=2, test=4.
class UnionFind:
    def __init__(self,n):self.parent=list(range(n));self.sz=[1]*n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]
            x=self.parent[x]
        return x
    def union(self,a,b):
        a=self.find(a);b=self.find(b)
        if a==b:return
        if self.sz[a]<self.sz[b]:a,b=b,a
        self.parent[b]=a;self.sz[a]+=self.sz[b]

def make_groups(data):
    uf=UnionFind(len(data))
    source_rep={};digest_rep={};digests=[]
    assert data['series_id'].notna().all() and (data['series_id'].astype(str).str.len()>0).all()
    for i,(rel,source) in enumerate(tqdm(
            zip(data.relative_path,data.series_id),total=len(data),desc='Audit source ID and exact-image SHA256')):
        source=str(source)
        if source in source_rep:uf.union(i,source_rep[source])
        else:source_rep[source]=i
        with (EXTRACT_DIR/rel).open('rb') as f:
            sha=hashlib.file_digest(f,'sha256').hexdigest()
        digests.append(sha)
        if sha in digest_rep:uf.union(i,digest_rep[sha])
        else:digest_rep[sha]=i
    groups=np.array([uf.find(i) for i in range(len(data))])
    print('GROUP AUDIT: unique series',len(source_rep),'unique SHA256',len(digest_rep),
          'disjoint groups',len(set(groups)))
    return groups,digests

split_path=DATA_DIR/'split_manifest.csv'
if split_path.exists() and not FORCE_RESPLIT:
    stored=pd.read_csv(split_path,dtype={'relative_path':str,'series_id':str})
    assert set(stored.relative_path)==set(df.relative_path),'Changed images: review required'
    assert stored[['relative_path','label']].set_index('relative_path').sort_index().equals(
        df[['relative_path','label']].set_index('relative_path').sort_index()),'Changed labels: review required'
    df=stored.sort_values('relative_path').reset_index(drop=True)
else:
    group,digests=make_groups(df)
    df=df.assign(group=group,sha256=digests)
    max_group=int(df.groupby('group').size().max())
    assert max_group<max(500,int(.04*len(df))),f'Oversized duplicate component {max_group}'
    skf=StratifiedGroupKFold(n_splits=10,shuffle=True,random_state=SEED_SPLIT)
    whole=np.arange(len(df))
    trainval_idx,test_idx=next(skf.split(whole,df.label,df.group))
    sg2=StratifiedGroupKFold(n_splits=9,shuffle=True,random_state=SEED_SPLIT+1)
    tr_sub,va_sub=next(sg2.split(trainval_idx,df.label.iloc[trainval_idx],df.group.iloc[trainval_idx]))
    df['split']='test'
    df.loc[trainval_idx[tr_sub],'split']='train'
    df.loc[trainval_idx[va_sub],'split']='val'
    df.to_csv(split_path,index=False)
assert df.groupby('group').split.nunique().max()==1,'SOURCE OR EXACT-DUPLICATE GROUP LEAKAGE'
assert df.relative_path.is_unique
for name,lower,upper in [('train',.74,.86),('val',.07,.14),('test',.07,.14)]:
    fraction=(df.split==name).mean()
    assert lower<fraction<upper, f'INVALID {name} FRACTION {fraction:.5f}'
    partition=df[df.split==name]
    assert set(partition.label.unique())=={0,1},f'Missing binary class: {name}'
    assert set(partition.category.unique())==set(CATEGORIES),f'Missing category: {name}'
assert df.groupby('sha256').split.nunique().max()==1,'EXACT BYTE-IDENTICAL IMAGE LEAKAGE'
assert df.groupby('series_id').split.nunique().max()==1,'NUMERICAL SOURCE SERIES LEAKAGE'
print('VALID SPLIT',df.split.value_counts().to_dict())
print('CLASS MATRIX\n',pd.crosstab(df.split,df.label))
print('CATEGORY MATRIX\n',pd.crosstab(df.split,df.category))
print('MANIFEST SHA256',hashlib.sha256(split_path.read_bytes()).hexdigest())
print('DUPLICATE GROUP COUNT',df.group.nunique())


## 5. Transforms, dataloaders, architecture

**SALF-DINO architecture**: same DINOv3 backbone as paper baseline, then (a) global patch mean, (b) structural-region token pooling over left, bottom and right border strips with learnable query attention, and (c) depthwise local token filtering with nonlinear activation. A gated residual fusion merges the structural/local representation with the global embedding and predicts two classes.

This is an architecture hypothesis, **not a proven novel method** until a prior-art review and ablations establish distinctiveness and value.

In [ ]:
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
seed_everything(SEED_SPLIT)

class MildWebCorruption:
    def __call__(self,img):
        p=random.random()
        if p<0.15: img=img.filter(ImageFilter.GaussianBlur(radius=random.uniform(.2,1.1)))
        elif p<0.30:
            buf=io.BytesIO();img.save(buf,format='JPEG',quality=random.randint(48,85))
            buf.seek(0)
            with Image.open(buf) as im: img=im.convert('RGB')
        return img

# Geometric flips are intentionally omitted: flipping can alter chart deception semantics.
def get_transform(size,training=False,corruption=None):
    steps=[T.Resize((size,size),interpolation=T.InterpolationMode.BICUBIC)]
    if corruption=='jpeg': steps.append(T.Lambda(lambda im: jpeg_corrupt(im,quality=40)))
    if corruption=='blur': steps.append(T.GaussianBlur(kernel_size=5,sigma=1.2))
    if corruption=='downsample': steps.append(T.Lambda(lambda im: im.resize((max(32,size//3),max(32,size//3))).resize((size,size))))
    if training and USE_ROBUST_TRAIN_AUG: steps.append(MildWebCorruption())
    return T.Compose(steps+[T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])
def jpeg_corrupt(img,quality):
    buff=io.BytesIO();img.save(buff,format='JPEG',quality=quality);buff.seek(0)
    with Image.open(buff) as im:return im.convert('RGB')
class ChartDataset(Dataset):
    def __init__(self,frame,transform):self.frame=frame.reset_index(drop=True);self.transform=transform
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        with Image.open(EXTRACT_DIR/r.relative_path) as im: img=im.convert('RGB')
        return self.transform(img),int(r.label),i

def make_loader(split, size, train=False, corruption=None):
    frame=df.loc[df.split==split].reset_index(drop=True)
    data=ChartDataset(frame,get_transform(size,training=train,corruption=corruption))
    return DataLoader(data,batch_size=BATCH_SIZE,shuffle=train,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda'),frame

class DinoTokens(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=timm.create_model(DINO_NAME,pretrained=True,img_size=RES,num_classes=0)
        self.dim=self.backbone.num_features
        self.prefix=int(getattr(self.backbone,'num_prefix_tokens',5))
    def forward(self,x):
        tokens=self.backbone.forward_features(x)
        if isinstance(tokens,(list,tuple)):tokens=tokens[0]
        if not torch.is_tensor(tokens) or tokens.ndim!=3:
            raise RuntimeError(f'Unexpected DINO token output {type(tokens)}. Check timm version.')
        patches=tokens[:,self.prefix:,:]
        side=math.isqrt(patches.shape[1])
        if side*side != patches.shape[1]:
            raise RuntimeError(f'Unexpected {patches.shape[1]} patch tokens; check prefix count and input size.')
        return patches,side

class PaperDINO(nn.Module):
    def __init__(self):
        super().__init__();self.encoder=DinoTokens()
        self.head=nn.Linear(self.encoder.dim,2)
    def forward(self,x):
        patch,_=self.encoder(x)
        return self.head(patch.mean(1))

class SALFDINO(nn.Module):
    def __init__(self, branch='both'):
        super().__init__();assert branch in ('both','axis_only','local_only')
        self.encoder=DinoTokens();d=self.encoder.dim;self.branch=branch
        self.query=nn.Linear(d,d,bias=False)
        self.key=nn.Linear(d,d,bias=False)
        self.local=nn.Sequential(nn.Conv2d(d,d,3,padding=1,groups=d),nn.GELU(),nn.Conv2d(d,d,1))
        self.project=nn.Linear(2*d,d)
        self.gate=nn.Linear(3*d,d)
        self.head=nn.Linear(d,2)
    @staticmethod
    def region_features(tokens,side):
        b,n,d=tokens.shape
        grid=tokens.reshape(b,side,side,d)
        w=max(1,round(side*0.20))
        return torch.stack([grid[:,:,:w,:].mean((1,2)),
                            grid[:,-w:,:,:].mean((1,2)),
                            grid[:,:,-w:,:].mean((1,2))],dim=1)
    def forward(self,x):
        patch,side=self.encoder(x)
        global_feature=patch.mean(1)
        region=self.region_features(patch,side)
        attention=torch.softmax((self.query(global_feature).unsqueeze(1)*self.key(region)).sum(-1)/math.sqrt(patch.shape[-1]),dim=1)
        axis_feature=(region*attention.unsqueeze(-1)).sum(1)
        local_feature=self.local(patch.reshape(x.shape[0],side,side,-1).permute(0,3,1,2)).mean((2,3))
        if self.branch=='axis_only':local_feature=torch.zeros_like(local_feature)
        if self.branch=='local_only':axis_feature=torch.zeros_like(axis_feature)
        features=torch.cat([global_feature,axis_feature,local_feature],dim=-1)
        correction=self.project(torch.cat([axis_feature,local_feature],dim=-1))
        refined=global_feature + torch.sigmoid(self.gate(features))*correction
        return self.head(refined)

class SwinV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=timm.create_model(SWIN_NAME,pretrained=True,num_classes=0)
        self.head=nn.Linear(self.encoder.num_features,2)
    def forward(self,x):return self.head(self.encoder(x))


class SALFContentMatched(SALFDINO):
    """Same layers and parameter count; replace border stripes with generic activation-ranked token groups."""
    @staticmethod
    def region_features(tokens,side):
        batch,n,d=tokens.shape
        assert n==side*side and n==256
        scores=tokens.detach().float().square().mean(dim=-1)
        order=torch.argsort(scores,dim=1,stable=True)
        groups=torch.tensor_split(order,3,dim=1)
        return torch.stack([torch.gather(tokens,1,indices.unsqueeze(-1).expand(-1,-1,d)).mean(1) for indices in groups],1)

class SALFAdditiveNoGate(SALFDINO):
    """Same trainable parameters; additive learned residual, without multiplicative sigmoid gate."""
    def forward(self,x):
        patches,side=self.encoder(x)
        g=patches.mean(1)
        region=self.region_features(patches,side)
        a=torch.softmax((self.query(g).unsqueeze(1)*self.key(region)).sum(-1)/math.sqrt(patches.shape[-1]),dim=1)
        border=(region*a.unsqueeze(-1)).sum(1)
        local=self.local(patches.reshape(x.shape[0],side,side,-1).permute(0,3,1,2)).mean((2,3))
        feat=torch.cat([g,border,local],dim=-1)
        correction=self.project(torch.cat([border,local],dim=-1))
        combined=g + correction + 0.5*torch.tanh(self.gate(feat))
        return self.head(combined)

MODEL_NAMES=['dinov3_global_avg','swinv2_tiny','proposed_salf']
if RUN_ABLATIONS:MODEL_NAMES += ['ablation_axis','ablation_local']
def build_model(name):
    if name=='dinov3_global_avg':return PaperDINO()
    if name=='swinv2_tiny':return SwinV2()
    if name=='proposed_salf':return SALFDINO('both')
    if name=='ablation_axis':return SALFDINO('axis_only')
    if name=='ablation_local':return SALFDINO('local_only')
    if name=='control_content_matched':return SALFContentMatched('both')
    if name=='control_additive_no_gate':return SALFAdditiveNoGate('both')
    raise KeyError(name)
def size_for(name):return 256 if name=='swinv2_tiny' else RES

## 6. Smoke test (architectural forward/backward)

This test constructs and backpropagates through just the **new local fusion layers**, without downloading 300M backbone weights. It does not replace actual end-to-end model training.

In [ ]:
def head_smoke_test():
    # Full SALF-DINO head structure is tested with realistic dimensions using a small fake encoder.
    class DummyEncoder(nn.Module):
        def __init__(self,d=32):super().__init__();self.dim=d;self.proj=nn.Conv2d(3,d,16,16)
        def forward(self,x):
            fmap=self.proj(x);return fmap.flatten(2).transpose(1,2),fmap.shape[-1]
    # Avoid loading pretrained weights by creating a stand-in module via __new__.
    net=SALFDINO.__new__(SALFDINO);nn.Module.__init__(net)
    d=32;net.encoder=DummyEncoder(d);net.branch='both'
    net.query=nn.Linear(d,d,bias=False);net.key=nn.Linear(d,d,bias=False)
    net.local=nn.Sequential(nn.Conv2d(d,d,3,padding=1,groups=d),nn.GELU(),nn.Conv2d(d,d,1))
    net.project=nn.Linear(2*d,d);net.gate=nn.Linear(3*d,d);net.head=nn.Linear(d,2)
    output=net(torch.randn(2,3,64,64))
    assert output.shape==(2,2) and torch.isfinite(output).all()
    output.sum().backward()
    assert net.query.weight.grad is not None and net.local[0].weight.grad is not None
    print('SALF head smoke test passed:',tuple(output.shape))
head_smoke_test()

## 7. Training and evaluation

The training loop uses AdamW, distinct backbone/head learning rates, label smoothing 0.2, grad clipping, cosine schedule and AMP. For `paper` mode: DINO-L, 384 px, 20 epochs, batch 12, full fine-tuning, warmup 2 epochs. SwinV2 is trained on the identical split, with its native 256 px input. For `pilot` mode: a DINOv3-Base *proxy*, not the published Large model, with frozen feature encoder and 2 epochs. Interpret pilot scores only as debugging results.

Model selection: **validation macro-F1**, never test set. The test set is accessed only after the best checkpoint is selected.

In [ ]:
def create_optimizer(model):
    if MODE=='pilot':
        for p in model.encoder.parameters():p.requires_grad_(False)
    head_params=[p for n,p in model.named_parameters() if p.requires_grad and not n.startswith('encoder.')]
    backbone_params=[p for n,p in model.named_parameters() if p.requires_grad and n.startswith('encoder.')]
    groups=[]
    if backbone_params:groups.append({'params':backbone_params,'lr':4e-5,'initial_lr':4e-5})
    if head_params:groups.append({'params':head_params,'lr':5e-4,'initial_lr':5e-4})
    assert groups
    opt=torch.optim.AdamW(groups,weight_decay=.05)
    def sch_fn(base):
        def multiplier(epoch):
            if epoch < min(2,EPOCHS):return (epoch+1)/min(2,EPOCHS)
            progress=(epoch+1-min(2,EPOCHS))/max(1,EPOCHS-min(2,EPOCHS))
            floor=1e-6/base
            return floor+(1-floor)*.5*(1+math.cos(math.pi*min(1.,progress)))
        return multiplier
    scheduler=torch.optim.lr_scheduler.LambdaLR(opt,[sch_fn(g['initial_lr']) for g in groups])
    return opt,scheduler

def epoch_train(model,loader,optimizer,amp_enabled):
    model.train()
    if MODE=='pilot':model.encoder.eval()  # keep frozen encoder deterministic
    scaler=torch.amp.GradScaler('cuda',enabled=amp_enabled)
    optimizer.zero_grad(set_to_none=True)
    total_loss,n=0.,0
    for step,(x,y,_) in enumerate(tqdm(loader,leave=False)):
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type,enabled=amp_enabled):
            logits=model(x)
            loss=F.cross_entropy(logits,y,label_smoothing=.2)
        total_loss+=loss.item()*y.numel();n+=y.numel()
        scaler.scale(loss/GRAD_ACCUM).backward()
        if (step+1)%GRAD_ACCUM==0 or step+1==len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.)
            scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
    return total_loss/max(1,n)

@torch.inference_mode()
def predict(model,loader):
    model.eval();ids=[];ytrue=[];yhat=[];prob=[]
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    t0=time.perf_counter()
    for x,y,i in tqdm(loader,leave=False):
        logits=model(x.to(DEVICE,non_blocking=True))
        p=logits.float().softmax(-1)[:,1].cpu().numpy()
        ids.extend(i.numpy().tolist());ytrue.extend(y.numpy().tolist());prob.extend(p.tolist())
        yhat.extend((p>=0.5).astype(int).tolist())
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    seconds=time.perf_counter()-t0
    return np.array(ids),np.array(ytrue),np.array(yhat),np.array(prob), seconds

def measure_metrics(y,pred):
    precision,recall,_,_=precision_recall_fscore_support(y,pred,average='macro',zero_division=0)
    return {'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),
            'macro_precision':precision,'macro_recall':recall,
            'macro_f1':f1_score(y,pred,average='macro',zero_division=0),
            'deceptive_recall':recall_score_binary(y,pred)}
def recall_score_binary(y,pred):
    tp=((y==1)&(pred==1)).sum();fn=((y==1)&(pred==0)).sum()
    return float(tp/max(1,tp+fn))

def do_experiment(name,seed):
    seed_everything(seed)
    model=build_model(name).to(DEVICE)
    optimizer,scheduler=create_optimizer(model)
    size=size_for(name)
    train_loader,_=make_loader('train',size,train=True)
    val_loader,_=make_loader('val',size)
    test_loader,test_frame=make_loader('test',size)
    amp_enabled=DEVICE.type=='cuda'
    best_f1=-1.;history=[];checkpoint=RESULT_DIR/f'{MODE}_{name}_{seed}_best.pt'
    for epoch in range(EPOCHS):
        loss=epoch_train(model,train_loader,optimizer,amp_enabled)
        ids,yt,yp,_,sec=predict(model,val_loader)
        metrics=measure_metrics(yt,yp)
        history.append({'epoch':epoch+1,'loss':loss,**metrics})
        print(name,seed,'epoch',epoch+1,'train loss',round(loss,4),'val F1',round(metrics['macro_f1'],4))
        if metrics['macro_f1']>best_f1:
            best_f1=metrics['macro_f1']
            torch.save({'model':model.state_dict(),'epoch':epoch+1,'best_val_f1':best_f1,
                        'mode':MODE,'seed':seed,'name':name,'resolution':size},checkpoint)
        scheduler.step()
    pd.DataFrame(history).to_csv(RESULT_DIR/f'{MODE}_{name}_{seed}_history.csv',index=False)
    ckpt=torch.load(checkpoint,map_location=DEVICE,weights_only=True)
    model.load_state_dict(ckpt['model'])
    idx,y,pred,p,seconds=predict(model,test_loader)
    assert np.array_equal(idx,np.arange(len(test_frame)))
    per_image=test_frame.copy();per_image['pred']=pred;per_image['prob_deceptive']=p
    per_image.to_csv(RESULT_DIR/f'{MODE}_{name}_{seed}_predictions.csv',index=False)
    metrics=measure_metrics(y,pred)
    metrics.update(model=name,seed=seed,mode=MODE,epochs=EPOCHS,resolution=size,
                   params_m=sum(v.numel() for v in model.parameters())/1e6,
                   eval_images_per_second=len(y)/max(seconds,1e-9),
                   val_best_macro_f1=best_f1)
    print('Held-out TEST:',metrics)
    cm=confusion_matrix(y,pred,labels=[0,1]);
    pd.DataFrame(cm,index=['true_0','true_1'],columns=['pred_0','pred_1']).to_csv(RESULT_DIR/f'{MODE}_{name}_{seed}_cm.csv')
    by_cat=[]
    for cat,group in per_image.groupby('category'):
        met=measure_metrics(group.label.to_numpy(),group.pred.to_numpy())
        by_cat.append({'category':cat,'n':len(group),**met})
    pd.DataFrame(by_cat).to_csv(RESULT_DIR/f'{MODE}_{name}_{seed}_by_category.csv',index=False)
    corrupt_rows=[]
    for corruption in ['jpeg','blur','downsample']:
        corrupted,_=make_loader('test',size,corruption=corruption)
        _,cy,cp,_,_=predict(model,corrupted)
        corrupt_rows.append({'corruption':corruption,**measure_metrics(cy,cp)})
    pd.DataFrame(corrupt_rows).to_csv(RESULT_DIR/f'{MODE}_{name}_{seed}_corruptions.csv',index=False)
    del model,optimizer,train_loader,val_loader,test_loader
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return metrics

# Crash-safe resume engine overrides the original training functions.
# It retains all three model architectures from the notebook itself.
# SALF-DINO fault-tolerant training engine. Executed from the notebook namespace.
import json, os, hashlib, random, time, gc
from pathlib import Path
AUTO_RESUME=True
EARLY_STOP_PATIENCE=5
EARLY_STOP_MIN_DELTA=0.0005
EARLY_STOP_MIN_EPOCHS=5
CHECKPOINT_DIR=ROOT/"checkpoints"
CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)

def create_optimizer(model):
    for p in model.encoder.parameters(): p.requires_grad_(False)
    head=[p for n,p in model.named_parameters() if not n.startswith("encoder.") and p.requires_grad]
    assert head, "No trainable head"
    opt=torch.optim.AdamW(head,lr=5e-4,weight_decay=0.05)
    def mult(epoch):
        warm=min(2,EPOCHS)
        if epoch<warm: return (epoch+1)/max(1,warm)
        progress=(epoch+1-warm)/max(1,EPOCHS-warm)
        return 0.002+0.998*0.5*(1+math.cos(math.pi*min(1.0,progress)))
    return opt,torch.optim.lr_scheduler.LambdaLR(opt,mult)

def epoch_train(model,loader,opt,scaler,amp_enabled):
    model.train();model.encoder.eval();opt.zero_grad(set_to_none=True)
    loss_total,n=0.,0
    for step,(x,y,_) in enumerate(tqdm(loader,leave=False)):
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type,enabled=amp_enabled):
            logits=model(x);loss=F.cross_entropy(logits,y,label_smoothing=0.2)
        loss_total+=loss.item()*y.numel();n+=y.numel()
        tail=len(loader)%GRAD_ACCUM
        group=GRAD_ACCUM if step<len(loader)-(tail or GRAD_ACCUM) else (tail or GRAD_ACCUM)
        scaler.scale(loss/group).backward()
        if (step+1)%GRAD_ACCUM==0 or step+1==len(loader):
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
            scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
    return loss_total/max(n,1)

def _head(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items() if not k.startswith("encoder.")}

def _load_head(model,weights):
    expected={k for k in model.state_dict() if not k.startswith("encoder.")}
    assert set(weights)==expected,"Checkpoint keys don't match"
    missing=model.load_state_dict(weights,strict=False)
    assert not missing.unexpected_keys and all(k.startswith("encoder.") for k in missing.missing_keys)

def _save(obj,path):
    tmp=path.with_name(path.name+".tmp")
    try: torch.save(obj,tmp);os.replace(tmp,path)
    finally:
        if tmp.exists(): tmp.unlink()

def _json(obj,path):
    tmp=path.with_name(path.name+".tmp")
    try:tmp.write_text(json.dumps(obj,indent=2),encoding="utf-8");os.replace(tmp,path)
    finally:
        if tmp.exists():tmp.unlink()

def _rng():
    npstate=np.random.get_state()
    return {"python":random.getstate(),
            "numpy":(str(npstate[0]),npstate[1].tolist(),int(npstate[2]),int(npstate[3]),float(npstate[4])),
            "torch":torch.get_rng_state().cpu(),
            "cuda":[x.cpu() for x in torch.cuda.get_rng_state_all()] if torch.cuda.is_available() else []}

def _restore_rng(r):
    random.setstate(r["python"])
    q=r["numpy"]
    np.random.set_state((q[0],np.array(q[1],dtype=np.uint32),q[2],q[3],q[4]))
    torch.set_rng_state(r["torch"].cpu())
    if torch.cuda.is_available() and r["cuda"]:torch.cuda.set_rng_state_all(r["cuda"])

def _signature(name,seed,size):
    split=DATA_DIR/"split_manifest.csv"
    if not split.exists():raise FileNotFoundError(split)
    config=dict(name=name,seed=int(seed),mode=MODE,epochs=EPOCHS,res=size,
                batch=BATCH_SIZE,grad_accum=GRAD_ACCUM,dino=DINO_NAME,swin=SWIN_NAME,
                split=hashlib.sha256(split.read_bytes()).hexdigest(),repo=DATASET_ID,
                aug=USE_ROBUST_TRAIN_AUG,patience=EARLY_STOP_PATIENCE,
                min_delta=EARLY_STOP_MIN_DELTA,min_epochs=EARLY_STOP_MIN_EPOCHS,
                training_protocol="frozen-backbone-resume-v2-kaggle")
    return hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest(),config

def _matching(obj,sig,path):
    if obj.get("signature")!=sig:
        raise RuntimeError("Checkpoint/config mismatch at "+str(path)+". Archive old run before changing parameters.")

def do_experiment(name,seed):
    size=size_for(name);sig,config=_signature(name,seed,size)
    prefix=f"{MODE}_{name}_{seed}"
    last=CHECKPOINT_DIR/(prefix+"_last.pt")
    best=CHECKPOINT_DIR/(prefix+"_best.pt")
    marker=RESULT_DIR/(prefix+"_completed.json")
    required=[RESULT_DIR/(prefix+"_"+ext+".csv") for ext in ("predictions","history","cm","by_category","corruptions")]
    if marker.exists():
        done=json.loads(marker.read_text(encoding="utf-8"))
        _matching(done,sig,marker)
        if all(p.is_file() for p in required):
            print("SKIP completed:",prefix)
            return done["metrics"]
    if last.exists() and not AUTO_RESUME:raise RuntimeError("Existing last.pt; resume or archive it")
    if best.exists() and not last.exists():raise RuntimeError("Orphan best.pt; archive before new experiment")

    seed_everything(seed)
    model=build_model(name).to(DEVICE)
    optimizer,scheduler=create_optimizer(model)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
    train_loader,_=make_loader("train",size,train=True)
    val_loader,_=make_loader("val",size)
    test_loader,test_frame=make_loader("test",size)
    start_epoch=0;history=[];best_f1=-1.;best_epoch=0;wait=0;stopped=False
    if last.exists():
        state=torch.load(last,map_location="cpu",weights_only=True)
        _matching(state,sig,last)
        _load_head(model,state["model"])
        optimizer.load_state_dict(state["optimizer"])
        for group in optimizer.state.values():
            for k,v in group.items():
                if torch.is_tensor(v):group[k]=v.to(DEVICE)
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        _restore_rng(state["rng"])
        start_epoch=state["epoch"];history=state["history"]
        best_f1=state["best_f1"];best_epoch=state["best_epoch"]
        wait=state["wait"];stopped=state["early_stopped"]
        assert start_epoch==len(history),"Saved history mismatch"
        print("RESUME",prefix,"NEXT EPOCH",start_epoch+1,"BEST",best_f1)
    for epoch in range(start_epoch,min(EPOCHS,start_epoch+MAX_EPOCHS_THIS_VERSION)):
        if stopped:break
        seed_everything(seed+100003*(epoch+1))
        loss=epoch_train(model,train_loader,optimizer,scaler,DEVICE.type=="cuda")
        _,yt,yp,_,_=predict(model,val_loader)
        metrics=measure_metrics(yt,yp);score=float(metrics["macro_f1"])
        if not np.isfinite(score):raise RuntimeError("Nonfinite validation score")
        history.append(dict(epoch=int(epoch+1),loss=float(loss),**{k:float(v) for k,v in metrics.items()}))
        if best_epoch==0 or score>best_f1+EARLY_STOP_MIN_DELTA:
            best_f1=score;best_epoch=epoch+1;wait=0
            _save(dict(model=_head(model),signature=sig,epoch=best_epoch,best_f1=best_f1),best)
        else:wait+=1
        scheduler.step()
        stopped=epoch+1>=EARLY_STOP_MIN_EPOCHS and wait>=EARLY_STOP_PATIENCE
        _save(dict(signature=sig,config=config,model=_head(model),
                   optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),
                   scaler=scaler.state_dict(),rng=_rng(),epoch=epoch+1,
                   history=history,best_f1=best_f1,best_epoch=best_epoch,
                   wait=wait,early_stopped=stopped),last)
        pd.DataFrame(history).to_csv(RESULT_DIR/(prefix+"_history.csv"),index=False)
        print(prefix,"epoch",epoch+1,"val macro-F1",round(score,5),"best",round(best_f1,5),"patience",wait)
        if stopped:print("EARLY STOP",prefix,"best epoch",best_epoch);break
    if len(history)<EPOCHS and not stopped:
        print('PARTIAL STAGE:',prefix,'saved through epoch',len(history),
              '; Save Version and attach that output before next stage.')
        del model,optimizer,scaler,train_loader,val_loader,test_loader
        gc.collect()
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
        return dict(model=name,seed=int(seed),status='partial',epochs_completed=len(history))
    if not best.exists():raise RuntimeError("Missing best.pt")
    winner=torch.load(best,map_location="cpu",weights_only=True)
    _matching(winner,sig,best)
    _load_head(model,winner["model"])
    idx,y,pred,prob,seconds=predict(model,test_loader)
    assert np.array_equal(idx,np.arange(len(test_frame)))
    per_image=test_frame.copy()
    per_image["pred"]=pred;per_image["prob_deceptive"]=prob
    per_image.to_csv(RESULT_DIR/(prefix+"_predictions.csv"),index=False)
    result=measure_metrics(y,pred)
    result.update(model=name,seed=int(seed),mode=MODE,epochs_completed=len(history),
                  best_epoch=best_epoch,early_stopped=bool(stopped),resolution=size,
                  params_m=sum(p.numel() for p in model.parameters())/1e6,
                  trainable_params_m=sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6,
                  eval_images_per_second=len(y)/max(seconds,1e-9),val_best_macro_f1=best_f1)
    pd.DataFrame(confusion_matrix(y,pred,labels=[0,1]),
                 index=["true_0","true_1"],columns=["pred_0","pred_1"]).to_csv(
                 RESULT_DIR/(prefix+"_cm.csv"))
    cat=[]
    for c,group in per_image.groupby("category"):
        cat.append(dict(category=c,n=len(group),**measure_metrics(group.label.to_numpy(),group.pred.to_numpy())))
    pd.DataFrame(cat).to_csv(RESULT_DIR/(prefix+"_by_category.csv"),index=False)
    distortions=[]
    for kind in ["jpeg","blur","downsample"]:
        loader,_=make_loader("test",size,corruption=kind)
        _,yt,yp,_,_=predict(model,loader)
        distortions.append(dict(corruption=kind,**measure_metrics(yt,yp)))
    pd.DataFrame(distortions).to_csv(RESULT_DIR/(prefix+"_corruptions.csv"),index=False)
    pd.DataFrame(history).to_csv(RESULT_DIR/(prefix+"_history.csv"),index=False)
    _json(dict(signature=sig,config=config,metrics=result),marker)
    print("COMPLETED",prefix,result)
    del model,optimizer,scaler,train_loader,val_loader,test_loader
    gc.collect()
    if DEVICE.type=="cuda":torch.cuda.empty_cache()
    return result



## 8. Run models on exactly the same split

Expected outputs: one checkpoint, training history, per-image predictions, confusion matrix, per-chart-type evaluation, and robustness CSV for each model and seed. Training may require substantial time and GPU memory in paper mode. No score is inferred if training has not actually run.

In [ ]:
# Cloud-only exact controlled trial
assert MODE=='paper' and EPOCHS==20 and MAX_EPOCHS_THIS_VERSION==20
assert RUN_TRAINING and torch.cuda.is_available() and DEVICE.type=='cuda'
assert EARLY_STOP_PATIENCE==5 and EARLY_STOP_MIN_DELTA==0.0005 and EARLY_STOP_MIN_EPOCHS==5
assert SELECTED_MODEL=='control_content_matched' and SELECTED_SEED==2025
assert hashlib.sha256((DATA_DIR/'split_manifest.csv').read_bytes()).hexdigest()=='c8fc29542853326611668612b0444b7c044853e9ca483c591fba65bbc86b4d7a'
model_probe=build_model(SELECTED_MODEL).to(DEVICE)
for p in model_probe.encoder.parameters(): p.requires_grad_(False)
assert not any(p.requires_grad for p in model_probe.encoder.parameters())
trainable=sum(p.numel() for p in model_probe.parameters() if p.requires_grad)
assert trainable==8403970,(trainable,SELECTED_MODEL)
del model_probe;torch.cuda.empty_cache()
print('CAPACITY_MATCHED_VERIFIED',SELECTED_MODEL,trainable,flush=True)
report=do_experiment('control_content_matched',2025)
assert report.get('model')=='control_content_matched' and int(report.get('seed'))==2025
assert report.get('status')!='partial',('Partial result: resume from exported checkpoints',report)
assert 5<=int(report['epochs_completed'])<=20
prefix='paper_control_content_matched_2025'
for suffix in ('completed.json','history.csv','predictions.csv','cm.csv','by_category.csv','corruptions.csv'):
 assert (RESULT_DIR/(prefix+'_'+suffix)).is_file(),suffix
print('EXPERIMENT_VERIFIED',report,flush=True)


## 9. Paired bootstrap comparison, figures, and scientific checks

A paired bootstrap estimates the uncertainty of the **proposed minus original-baseline macro-F1** on identical held-out items. Group-resampling by source series is preferable when authoritative series IDs exist; this notebook performs item-level bootstrap, which may be overconfident if hidden source-series dependence remains. Re-run the same pipeline with verified source groups for a publishable confidence interval.

In [ ]:
"""Paired group bootstrap: same held-out series/near-duplicate groups in both arms."""
import numpy as np
from sklearn.metrics import f1_score

def paired_group_bootstrap(baseline, proposed, group_key="group", replicates=2000, seed=2026):
    required=["relative_path","label","pred",group_key]
    for frame in (baseline,proposed):
        assert set(required).issubset(frame.columns),set(required)-set(frame.columns)
    assert list(baseline["relative_path"])==list(proposed["relative_path"])
    assert np.array_equal(baseline["label"].to_numpy(),proposed["label"].to_numpy())
    assert np.array_equal(baseline[group_key].to_numpy(),proposed[group_key].to_numpy())
    true=baseline["label"].to_numpy(dtype=int)
    b=baseline["pred"].to_numpy(dtype=int)
    p=proposed["pred"].to_numpy(dtype=int)
    groups=baseline[group_key].to_numpy()
    uniq=np.unique(groups)
    assert len(uniq)>=2,"Need more than one independent group"
    positions={g:np.where(groups==g)[0] for g in uniq}
    def f(y,z):return f1_score(y,z,labels=[0,1],average="macro",zero_division=0)
    delta=float(f(true,p)-f(true,b))
    rng=np.random.default_rng(seed)
    draws=np.empty(replicates,dtype=float)
    for k in range(replicates):
        selected=rng.choice(uniq,size=len(uniq),replace=True)
        indices=np.concatenate([positions[g] for g in selected])
        draws[k]=float(f(true[indices],p[indices])-f(true[indices],b[indices]))
    lo,hi=np.quantile(draws,[0.025,0.975])
    return delta,float(lo),float(hi),len(uniq)

# Read completed model results only, with no fabricated score.
import pandas as pd,json
records=[]
for file in RESULT_DIR.glob(f'{MODE}_*_completed.json'):
    obj=json.loads(file.read_text(encoding='utf-8'))
    if 'metrics' in obj:records.append(obj['metrics'])
if records:
    results=pd.DataFrame(records).sort_values(['model','seed'])
    results.to_csv(RESULT_DIR/f'{MODE}_comparison.csv',index=False)
    display(results)
seed=SELECTED_SEED
b=RESULT_DIR/f'{MODE}_dinov3_global_avg_{seed}_predictions.csv'
p=RESULT_DIR/f'{MODE}_proposed_salf_{seed}_predictions.csv'
if b.exists() and p.exists():
    base=pd.read_csv(b);prop=pd.read_csv(p)
    diff,low,high,n=paired_group_bootstrap(base,prop,group_key='group',replicates=2000)
    print('Paired group bootstrap macro-F1 change:',diff,'CI:',(low,high),'groups',n)
else:
    print('Baseline and SALF-DINO predictions not both complete; no comparative claim.')


## 10. Interpretation and remaining gates before IEEE Access submission

**The notebook only establishes a legitimate controlled comparison if the following pass:** archive image count and label audit; manually inspected class names; group-source leakage audit; shared split; all three models trained with real pretrained weights; 3 seeds for paper mode; matched DINO settings; reported test F1, deceptive recall, per-category `srtAxY` error and corrupted-image F1; ablations with and without axis and local branches; accuracy-vs-compute profile; statistical confidence intervals. Only then can one claim observed improvements.

**External web-mining claim is not tested by DVCS**. To make the paper a web-mining study, add a manually verified real-web chart subset with source URLs, image licenses, sampling protocol, source-group disjointness and blinded dual-annotator labels. Keep it **external** to HVPD and do not use its test labels to tune the method. A real-web study is distinct from the controlled HVPD comparison.

**Citation integrity:** Retain separate columns for `published baseline (from paper)`, `reproduced baseline (this protocol)`, `SwinV2 comparator`, and `SALF-DINO proposed`. Never claim the reproduction is the same original experiment merely because model family and dataset match.

## Output management

After Kaggle Save & Run All finishes, export all paper_proposed_salf_42_* results and the checkpoints paper_proposed_salf_42_best.pt and paper_proposed_salf_42_last.pt files to D: before any follow-up version. Do not mix with laptop SwinV2 or other seeds.
